In [ ]:
import sys
sys.path.append('..')
from pathlib import Path

from tools.geometry import generate_detector
from tools.utils import generate_random_params
from tools.utils import load_single_event, save_single_event
import jax
import jax.numpy as jnp
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim
from tools.utils import spherical_to_cartesian

# Generate and save a single event
key = jax.random.PRNGKey(6)

detector_params = (
    jnp.array(100),         # scattering_length
    jnp.array(0.2),         # reflection_rate
    jnp.array(1000),        # absorption_length
    jnp.array(0.001)        # deprecated
)

track_params = (
    jnp.array(1050.0, dtype=jnp.float32),              # energy 
    jnp.array([0.0, 0.0, 0.0], dtype=jnp.float32),     # position
    jnp.array([jnp.pi/2, jnp.pi/6], dtype=jnp.float32)
)

figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def visualize_detector_geometry(detector_name, colorscale='plasma', surface_color='black'):
    """
    Visualize detector geometry with all sensors having equal charge (same color)
    """
    json_filename = f'../config/{detector_name}_geom_config.json'
    detector = generate_detector(json_filename)
    
    # Create equal charges for all sensors (all = 1)
    n_sensors = len(detector.all_points)
    indices = jnp.arange(n_sensors)
    charges = jnp.ones(n_sensors)  # All charges equal to 1
    times = jnp.zeros(n_sensors)   # Times don't matter for geometry visualization
    
    print(f"Detector: {detector_name}")
    print(f"Number of sensors: {n_sensors}")
    
    # Create visualization
    figname = f'figures/{detector_name}_geometry.pdf'
    detector.visualize_event_data_plotly_discs(
        indices, charges, times, 
        show_all_sensors=True, 
        log_scale=False,  # No need for log scale with equal charges
        show_colorbar=False, 
        dark_theme=False, 
        plot_time=False, 
        colorscale=colorscale, 
        surface_color=surface_color, 
        figname=figname
    )
    
    return detector

In [ ]:
# Visualize multiple detectors (comment out detectors you don't want to see)
for detector_name in ['MidBox', 'IWCD', 'SK', 'HK', 'JUNO']:  # Add more detector names as needed
    try:
        print(f"\nVisualizing {detector_name}...")
        visualize_detector_geometry(detector_name, colorscale='plasma', surface_color='black')
        print(f"Saved: figures/{detector_name}_geometry.pdf")
    except Exception as e:
        print(f"Error visualizing {detector_name}: {e}")

In [ ]:
detector_names = ['SK', 'JUNO', 'MidBox']

In [ ]:
generate_event = True
use_calibration = False

if generate_event:
    for name in detector_names:
        json_filename = f'../config/{name}_geom_config.json'
        detector = generate_detector(json_filename)
        detector_points = jnp.array(detector.all_points)
        Nphot = 1_000_000
        temperature = 0.0
        generate_event = False
        
        detector_type= None
        if name == 'TAO' or name == 'JUNO':
            detector_type='Sphere'
        elif 'Box' in name or 'nuSCOPE' in name:
            detector_type='Box'
        else:
            detector_type='Cylinder'
        

        simulator = None
        single_event = None
        if use_calibration is False:
            simulator = setup_event_simulator(json_filename, Nphot, temperature=temperature, K=6, is_data=False, is_calibration=False, detector_type=detector_type, max_sensors_per_cell=4)
            single_event = jax.lax.stop_gradient(simulator(track_params, detector_params, key))
        else:
            source_params = (
                jnp.array([0.0, 0.0, 0.0], dtype=jnp.float32),
                jnp.array(1.0, dtype=jnp.float32)
            )
            simulator = setup_event_simulator(json_filename, Nphot, temperature=temperature, K=2, is_data=False, is_calibration=True, detector_type=detector_type, max_sensors_per_cell=4)
            single_event = jax.lax.stop_gradient(simulator(source_params, detector_params, key))

        # Create events folder if it doesn't exist
        output_dir = Path('output/')
        output_dir.mkdir(parents=True, exist_ok=True)
    
        save_single_event(single_event, track_params, detector_params, filename=f'output/{name}_pred_event.h5', calibration_mode=False)

In [ ]:
# Generate and save a DATA event
generate_data_event = True
use_data = True

if generate_data_event:
    for name in detector_names:
        json_filename = f'../config/{name}_geom_config.json'
        detector = generate_detector(json_filename)
        detector_points = jnp.array(detector.all_points)
        Nphot = 1_000_000
        temperature = 0.0  # Zero temperature for data mode
        
        detector_type = None
        if name == 'TAO' or name == 'JUNO':
            detector_type = 'Sphere'
        elif 'Box' in name or 'nuSCOPE' in name:
            detector_type = 'Box'
        else:
            detector_type = 'Cylinder'
        
        # Setup data simulator
        data_simulator = setup_event_simulator(
            json_filename, Nphot, 
            temperature=temperature, 
            K=6, 
            is_data=True,  # Key difference: use data mode
            is_calibration=False, 
            detector_type=detector_type, 
            max_sensors_per_cell=4
        )
        
        # Load photon data from ROOT file
        data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
        entry_idx = 0  # Use first entry
        photon_data = read_photon_data_from_photonsim(data_file, entry_idx)
        photon_data['N'] = len(photon_data['photon_origins'])
        
        # Use same track parameters but convert for data simulator
        energy = track_params[0]
        position = track_params[1]
        angles = track_params[2]
        
        # Convert angles to direction vector
        theta, phi = angles
        direction = spherical_to_cartesian(theta, phi)
        
        # Create data track params (energy, position, direction vector)
        data_track_params = (energy, position, direction)
        
        # Generate data event
        data_event_key = jax.random.PRNGKey(42)  # Different seed
        data_single_event = jax.lax.stop_gradient(
            data_simulator(data_track_params, detector_params, data_event_key, photon_data)
        )
        
        # Save data event
        save_single_event(
            data_single_event, 
            track_params,  # Save with original format for consistency
            detector_params, 
            filename=f'output/{name}_data_like_event.h5', 
            calibration_mode=False
        )
        
        print(f"Data event generated and saved for {name}")
        print(f"  Active sensors: {jnp.sum(data_single_event[0] > 0)}")
        print(f"  Total charge: {jnp.sum(data_single_event[0]):.1f}")

In [ ]:
def visualize_3D_event_for_detector(name, colorscale='viridis', surface_color='gray'):
    """Visualize predicted event using disc visualization"""
    _, _, indices, charges, times = load_single_event(
        f'output/{name}_pred_event.h5', 
        None, 
        calibration_mode=False
    )
    json_filename = f'../config/{name}_geom_config.json'
    detector = generate_detector(json_filename)
    figname = f'figures/{name}_3D_event_prediction.pdf'
    
    detector.visualize_event_data_plotly_discs(
        indices, 
        charges, 
        times, 
        show_all_sensors=True, 
        log_scale=True, 
        show_colorbar=False, 
        dark_theme=False, 
        plot_time=False, 
        colorscale=colorscale, 
        surface_color=surface_color, 
        figname=figname
    )

def visualize_3D_data_event_for_detector(name, colorscale='viridis', surface_color='gray'):
    """Visualize data event using disc visualization"""
    _, _, indices, charges, times = load_single_event(
        f'output/{name}_data_like_event.h5', 
        None, 
        calibration_mode=False
    )
    json_filename = f'../config/{name}_geom_config.json'
    detector = generate_detector(json_filename)
    figname = f'figures/{name}_3D_data_like_event.pdf'
    
    detector.visualize_event_data_plotly_discs(
        indices, 
        charges, 
        times, 
        show_all_sensors=True, 
        log_scale=True, 
        show_colorbar=False, 
        dark_theme=False, 
        plot_time=False, 
        colorscale=colorscale, 
        surface_color=surface_color, 
        figname=figname,
        title=f'{name} - Data Event'
    )

In [ ]:
def check_missing_sensors(name):
    _, _, indices, charges, times = load_single_event(f'output/{name}_pred_event.h5', None, calibration_mode=False)
    json_filename = f'../config/{name}_geom_config.json'
    detector = generate_detector(json_filename)
    if len(detector.all_points) == len(indices):
        print(f'No Missing Sensors in {name}!')
    else:
        print('Sensors Missing:')
        print(len(detector.all_points), len(indices))

for name in detector_names:
    check_missing_sensors(name)

In [ ]:
visualize_3D_event_for_detector('SK', surface_color='black', colorscale='viridis')
visualize_3D_data_event_for_detector('SK', surface_color='black', colorscale='viridis')